# 🌿 MangroveNet — Cloud Training on Google Colab

This notebook trains the **TCCFNet** (Transformer-CNN Combined Feature Network) model for mangrove vegetation segmentation using a **free T4 GPU** on Google Colab.

## Reproducible tuning experiment
- Run the existing configuration as a baseline.
- Run a candidate with differential learning rates, longer warmup, shorter backbone freeze, and a higher Dice-loss weight.
- Sweep the validation threshold from 0.50 to 0.90 and select the threshold with maximum validation IoU.
- Compare both logs using IoU, Dice, precision, recall, and the pixel confusion matrix.

The exact rationale and result table are tracked in `TRAINING_TUNING.md`.


## 1. Setup — Clone Repo & Install Dependencies

In [ ]:
# Clone the repository
!git clone https://github.com/prajwal-gowd/magrove-mapping.git
%cd magrove-mapping

# Install dependencies (GPU version — Colab already has PyTorch)
!pip install -q fastapi uvicorn python-multipart Pillow scikit-learn matplotlib tqdm seaborn

In [ ]:
# Verify GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

## 2. Upload Dataset

Upload your `dataset_patches/` folder. You have two options:

**Option A: Google Drive (Recommended)**
1. Upload `dataset_patches.zip` to your Google Drive
2. Run the cell below to mount Drive and extract

**Option B: Direct Upload**
- Use the Colab file browser to upload `dataset_patches.zip` directly

In [ ]:
# Option A: Mount Google Drive and copy dataset
from google.colab import drive
drive.mount('/content/drive')

# Adjust this path to where you stored your zip file in Drive
DRIVE_DATASET_PATH = '/content/drive/MyDrive/dataset_patches.zip'

import os
if os.path.exists(DRIVE_DATASET_PATH):
    !unzip -q {DRIVE_DATASET_PATH} -d .
    print('Dataset extracted successfully!')
else:
    print(f'Dataset not found at {DRIVE_DATASET_PATH}')
    print('Please upload dataset_patches.zip to your Google Drive and update the path above.')

In [ ]:
# Verify dataset structure
import os
for split in ['train', 'val']:
    img_dir = f'dataset_patches/{split}/images'
    mask_dir = f'dataset_patches/{split}/masks'
    if os.path.exists(img_dir):
        n_imgs = len(os.listdir(img_dir))
        n_masks = len(os.listdir(mask_dir))
        print(f'{split}: {n_imgs} images, {n_masks} masks')
    else:
        print(f'{split}: NOT FOUND at {img_dir}')

## 3. Verify Config & Model

In [ ]:
import config

print('=== Training Configuration ===')
print(f'Image Size:           {config.IMAGE_SIZE}')
print(f'Batch Size:           {config.BATCH_SIZE}')
print(f'Epochs:               {config.NUM_EPOCHS}')
print(f'Learning Rate:        {config.LEARNING_RATE}')
print(f'Backbone LR:          {config.BACKBONE_LR}')
print(f'Weight Decay:         {config.WEIGHT_DECAY}')
print(f'Warmup Epochs:        {config.WARMUP_EPOCHS}')
print(f'Backbone Freeze:      {config.BACKBONE_FREEZE_EPOCHS} epochs')
print(f'Gradient Clip:        {config.GRAD_CLIP_MAX_NORM}')
print(f'Decoder Dropout:      {config.DECODER_DROPOUT}')
print(f'Loss Weights:         BCE={config.BCE_WEIGHT}, Dice={config.DICE_WEIGHT}')
print(f'Focal Alpha/Gamma:    {config.FOCAL_ALPHA} / {config.FOCAL_GAMMA}')
print(f'Device:               {config.DEVICE}')

In [ ]:
# Test model forward pass
from models.hybrid_model import HybridTCCFNet

model = HybridTCCFNet(num_classes=1)
dummy = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    out = model(dummy)
print(f'Model output shape: {out.shape}  (expected: [2, 1, 224, 224])')

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
del model, dummy, out

## 4. Train the Model 🚀

This will run for **50 epochs** with all improvements enabled.

Expected time on T4 GPU: ~30-45 minutes (depends on dataset size).

In [ ]:
# Run training (fresh start)
!python train.py 2>&1 | tee training.log

In [ ]:
# OR: Resume from a previous checkpoint (uncomment to use)
# !python train.py --resume 2>&1 | tee training_resumed.log

## 4b. Retrain the tuned candidate 🚀

This run keeps the baseline checkpoint separate and tests the candidate settings recorded in `TRAINING_TUNING.md`.


In [ ]:
# Candidate: faster decoder adaptation, protected backbone, stronger overlap objective
!python train.py \
  --batch-size 16 \
  --lr 2e-4 \
  --backbone-lr 2e-5 \
  --epochs 60 \
  --warmup-epochs 5 \
  --freeze-epochs 3 \
  --bce-weight 0.3 \
  --dice-weight 0.7 \
  --threshold 0.75 \
  --save-path checkpoints/tuned_model.pth \
  2>&1 | tee training_tuned.log

## 4c. Select the validation threshold

The threshold is selected only from the validation split. This addresses the previous run’s high false-positive count without changing the underlying model weights.


In [ ]:
import torch
from torch.utils.data import DataLoader
from models.hybrid_model import HybridTCCFNet
from utils.dataloader import SegmentationDataset
import config

thresholds = [round(x / 100, 2) for x in range(50, 91, 5)]
dataset = SegmentationDataset(config.DATA_DIR, split="val", is_train=False)
loader = DataLoader(dataset, batch_size=config.BATCH_SIZE, shuffle=False, num_workers=2)
model = HybridTCCFNet(num_classes=config.NUM_CLASSES, pretrained_backbone=False)
checkpoint = torch.load("checkpoints/tuned_model.pth", map_location=config.DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"] if "model_state_dict" in checkpoint else checkpoint)
model.to(config.DEVICE).eval()

counts = {t: {"tp": 0, "fp": 0, "fn": 0} for t in thresholds}
with torch.no_grad():
    for images, masks in loader:
        probabilities = torch.sigmoid(model(images.to(config.DEVICE))).cpu()
        masks = masks.cpu()
        for threshold in thresholds:
            preds = probabilities > threshold
            counts[threshold]["tp"] += int((preds & (masks > 0.5)).sum())
            counts[threshold]["fp"] += int((preds & (masks <= 0.5)).sum())
            counts[threshold]["fn"] += int((~preds & (masks > 0.5)).sum())

results = []
for threshold, c in counts.items():
    tp, fp, fn = c["tp"], c["fp"], c["fn"]
    iou = tp / max(tp + fp + fn, 1)
    dice = 2 * tp / max(2 * tp + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    results.append((threshold, iou, dice, precision, recall))

results.sort(key=lambda row: row[1], reverse=True)
print("threshold | IoU   | Dice  | precision | recall")
for row in results:
    print(f"{row[0]:9.2f} | {row[1]:.4f} | {row[2]:.4f} | {row[3]:.4f}    | {row[4]:.4f}")
print(f"Selected threshold: {results[0][0]:.2f} (IoU={results[0][1]:.4f})")

## 5. Evaluate — Plot Training Curves

In [ ]:
import re
import matplotlib.pyplot as plt

# Parse training log
with open('training.log', 'r') as f:
    log_content = f.read()

# Extract metrics
train_losses = [float(x) for x in re.findall(r'Train Loss: ([\d.]+)', log_content)]
val_losses = [float(x) for x in re.findall(r'Val Loss: ([\d.]+)', log_content)]
val_ious = [float(x) for x in re.findall(r'Val IoU: ([\d.]+)', log_content)]
val_dices = [float(x) for x in re.findall(r'Val Dice: ([\d.]+)', log_content)]
val_precisions = [float(x) for x in re.findall(r'Val Precision: ([\d.]+)', log_content)]
val_recalls = [float(x) for x in re.findall(r'Val Recall: ([\d.]+)', log_content)]

epochs = range(1, len(train_losses) + 1)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('MangroveNet Training Progress', fontsize=16, fontweight='bold')

# Loss
axes[0, 0].plot(epochs, train_losses, 'b-o', markersize=3, label='Train Loss')
if val_losses:
    axes[0, 0].plot(epochs[:len(val_losses)], val_losses, 'r-o', markersize=3, label='Val Loss')
axes[0, 0].set_title('Loss'); axes[0, 0].set_xlabel('Epoch'); axes[0, 0].legend(); axes[0, 0].grid(True, alpha=0.3)

# IoU & Dice
if val_ious:
    axes[0, 1].plot(epochs[:len(val_ious)], val_ious, 'g-o', markersize=3, label='Val IoU')
if val_dices:
    axes[0, 1].plot(epochs[:len(val_dices)], val_dices, 'm-o', markersize=3, label='Val Dice')
axes[0, 1].set_title('IoU & Dice'); axes[0, 1].set_xlabel('Epoch'); axes[0, 1].legend(); axes[0, 1].grid(True, alpha=0.3)

# Precision & Recall
if val_precisions:
    axes[1, 0].plot(epochs[:len(val_precisions)], val_precisions, 'c-o', markersize=3, label='Val Precision')
if val_recalls:
    axes[1, 0].plot(epochs[:len(val_recalls)], val_recalls, 'y-o', markersize=3, label='Val Recall')
axes[1, 0].set_title('Precision & Recall'); axes[1, 0].set_xlabel('Epoch'); axes[1, 0].legend(); axes[1, 0].grid(True, alpha=0.3)

# Best metrics summary
if val_ious:
    best_epoch = val_ious.index(max(val_ious)) + 1
    summary = (f'Best Epoch: {best_epoch}\n'
               f'Best IoU: {max(val_ious):.4f}\n'
               f'Best Dice: {max(val_dices):.4f}\n')
    if val_precisions:
        summary += f'Precision @ Best: {val_precisions[best_epoch-1]:.4f}\n'
        summary += f'Recall @ Best: {val_recalls[best_epoch-1]:.4f}'
    axes[1, 1].text(0.5, 0.5, summary, transform=axes[1, 1].transAxes,
                    fontsize=14, verticalalignment='center', horizontalalignment='center',
                    fontfamily='monospace',
                    bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))
axes[1, 1].set_title('Best Results'); axes[1, 1].axis('off')

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')

## 6. Test Inference on Sample Image

In [ ]:
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from utils.inference import load_model_checkpoint, predict
from models.hybrid_model import HybridTCCFNet
import config

# Load trained model
model = HybridTCCFNet(pretrained_backbone=False)
model = load_model_checkpoint(model, "checkpoints/tuned_model.pth" if os.path.exists("checkpoints/tuned_model.pth") else config.SAVE_PATH, config.DEVICE)

# Pick a validation sample
import os
val_images = sorted(os.listdir('dataset_patches/val/images'))
sample_name = val_images[0]
sample_path = f'dataset_patches/val/images/{sample_name}'
mask_path = f'dataset_patches/val/masks/{sample_name}'

# Predict
pred_mask = predict(model, sample_path, config.DEVICE)

# Display
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(Image.open(sample_path)); axes[0].set_title('Input')
axes[1].imshow(Image.open(mask_path), cmap='gray'); axes[1].set_title('Ground Truth')
axes[2].imshow(pred_mask, cmap='gray'); axes[2].set_title('Predicted Mask')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.show()

## 7. Download Trained Checkpoint

In [ ]:
# Option A: Save to Google Drive
import shutil
drive_save_path = '/content/drive/MyDrive/mangrove_checkpoints/'
os.makedirs(drive_save_path, exist_ok=True)
if os.path.exists('checkpoints/tuned_model.pth'):
    shutil.copy('checkpoints/tuned_model.pth', drive_save_path)
shutil.copy('checkpoints/best_model.pth', drive_save_path)
shutil.copy('training.log', drive_save_path)
if os.path.exists('training_curves.png'):
    shutil.copy('training_curves.png', drive_save_path)
print(f'Checkpoint saved to {drive_save_path}')

In [ ]:
# Option B: Download directly to your computer
from google.colab import files
files.download('checkpoints/best_model.pth')

## 8. Generate Confusion Matrix

In [ ]:
!python generate_confusion_matrix.py

from IPython.display import Image as IPImage, display
if os.path.exists('confusion_matrix.png'):
    display(IPImage('confusion_matrix.png'))